# My Notes: Getting Started with Hugging Face Pipelines

I'm starting my journey into LLM engineering. The Hugging Face `transformers` library is my primary toolset for open-source AI.

**Pipelines** are my entry point. They handle the heavy lifting (tokenization, loading models, post-processing) automatically.

### My Workflow:
1. **Initialize**: `my_pipeline = pipeline("task-name")` (Creates the object)
2. **Run**: `result = my_pipeline("input")` (Gets the answer)

## Core Concept: Inference vs. Training

I need to distinguish between these two phases to understand how I'll be interacting with models:

### 1. Training & Fine-Tuning
This is the 'learning' phase. **Training** is starting from zero. **Fine-tuning** is taking a model that already knows things (pre-trained) and giving it specific knowledge from a smaller dataset of my own.

### 2. Inference
This is the 'using' phase. When I run a model to get a prediction or generation, I'm doing inference. **The Pipelines API is my go-to for easy inference.**

In [ ]:
# Installing the stack I need for my LLM projects
# - transformers: the core library for LLM work
# - datasets: for loading the data I'll use to test/train
# - accelerate: ensures I'm using the GPU efficiently
!pip install -q --upgrade datasets transformers diffusers accelerate

In [ ]:
# I always check for GPU availability first to ensure fast processing.
# T4 is great for these experiments.
import torch
if torch.cuda.is_available():
    print(f"GPU is active and ready: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Running on CPU. I should probably switch to a GPU runtime.")

In [ ]:
# Imports
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

## My Guide to Using Pipelines

Pipelines help me run inference for common tasks without having to manually set up tokenizers and configs every time.

**Step 1: Setup**
I'll specify the `task`, pick a `model` if I want more control, and set the `device` (always use 'cuda' if I have a GPU).

**Step 2: Execution**
I simply pass my input data into the pipeline object I created.

In [ ]:
# My first test: Sentiment Analysis
# Defaulting to 'cuda' to make it snappy.
classifier = pipeline("sentiment-analysis", device="cuda" if torch.cuda.is_available() else "cpu")

# Checking how the model handles a basic string
result = classifier("I am learning how to use Hugging Face and it's amazing!")
print(f"Result: {result}")

In [ ]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

In [ ]:
# I'm trying a specific multilingual model to see if I get better granularity.
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
# Exploring NER (Named Entity Recognition)
# I want to see how the model extracts organizations and locations.
ner_pipe = pipeline("ner", device="cuda" if torch.cuda.is_available() else "cpu")
text = "Hugging Face is based in New York and was founded by Clement Delangue."
for entity in ner_pipe(text):
    print(entity)

In [ ]:
# Testing Question Answering capabilities
# I need to provide 'context' so the model knows where to look for the answer.
qa_pipe = pipeline("question-answering", device="cuda" if torch.cuda.is_available() else "cpu")
context = "Pipelines are the simplest way to use models for inference."
question = "What are pipelines used for?"

result = qa_pipe(question=question, context=context)
print(f"Answer found: {result['answer']}")

In [ ]:
# Summarization test
# I can tweak max_length and min_length to control how concise I want my summary to be.
summarizer = pipeline("summarization", device="cuda" if torch.cuda.is_available() else "cpu")

long_text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

summary = summarizer(long_text, max_length=50, min_length=25, do_sample=False)
print(f"My Summary: {summary[0]['summary_text']}")

In [ ]:
# Testing Translation
# I'll try English to French first.
translator = pipeline("translation_en_to_fr", device="cuda" if torch.cuda.is_available() else "cpu")
result = translator("Learning AI is a journey, not a destination.")
print(f"French Translation: {result[0]['translation_text']}")

In [ ]:
# Trying a specific translation model from the Helsinki-NLP group for Spanish.
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(f"Spanish Translation: {result[0]['translation_text']}")

In [ ]:
# Zero-shot Classification
# This is powerful because I don't need to retrain the model for my specific labels.
classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)

In [ ]:
# Text Generation experiment
# I'll see how it completes a thought about pipelines.
generator = pipeline("text-generation", device="cuda")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])

In [ ]:
# Image Generation using Diffusion
# I'm using the pipeline approach for image generation too.
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Text-to-Speech (TTS) experiment
# I need a speaker embedding to give the voice a specific tone.
synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

# My Learning Resources

To continue my journey, I should bookmark these:

- [Transformers Pipeline Docs](https://huggingface.co/docs/transformers/main_classes/pipelines)
- [The Model Hub](https://huggingface.co/models) - This is where I find specific models for my projects.